# PBH Memory Burden Constraints — CMB + BBN

**Paper**: "Does Memory Burden Open a New Mass Window for PBHs as Dark Matter?"

This notebook computes CMB and BBN constraints on primordial black holes (PBHs) incorporating the **memory burden effect** — a conjectured slowdown of Hawking evaporation when a black hole's Page entropy becomes comparable to its Bekenstein-Hawking entropy.

## Key features
- **Three-phase CMB constraint**: SC (semi-classical) + Transition + MB (memory-burden) phases
- **BBN constraint** with memory-burden phase scan (Supplemental Material S1)
- **Combined CMB+BBN** constraint (most restrictive at each mass)
- **Four transition functions** validation (h1–h4, additive & multiplicative distributions)



In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
from scipy.interpolate import interp1d
from scipy.special import erf
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully.")


## Physical Constants

All constants in cgs units. The evaporation coefficient $\beta$ is chosen to reproduce the standard Page evaporation time $t_{\rm evap} \approx M^3 / (3\beta)$.


In [ ]:
# ==================== Physical Constants ====================
C_LIGHT = 2.998e10          # speed of light [cm/s]
M_Pl = 2.176e-5             # Planck mass [g]
evap_coef = 8.2e6 * (1e10)**2   # evaporation coefficient β [g³/s] = 8.2e26
S_ref = 2.6e30              # reference Page number
M_ref = 1e10                # reference mass [g]
Omega_DM = 0.26             # DM density parameter
rho_c = 9.2e-30             # critical density [g/cm³]
rho_DM_ref = 3.6e-18        # reference DM density [g/cm³]
Lambda_CMB = 4.7e-34 / 1.9e-39  # CMB constraint ratio

print("Physical constants set.")
print(f"  evap_coef = {evap_coef:.2e} g³/s")
print(f"  S_ref     = {S_ref:.2e}")
print(f"  M_ref     = {M_ref:.2e} g")


## Acharya+2019 CMB Data

Complete 21-point dataset from Fig. 9 of Acharya *et al.* 2019 [arXiv:1905.00015], giving the CMB upper limit on $f_{\rm PBH}$ as a function of decaying-particle lifetime $\tau_X$.


In [ ]:
# ==================== Acharya+2019 Fig.9 CMB Data (21 points) ====================
ACHARYA_CMB_DATA = np.array([
    [1.71e11, 4.39e-2],
    [2.30e11, 5.84e-3],
    [3.34e11, 5.34e-4],
    [3.88e11, 2.72e-4],
    [5.23e11, 3.36e-5],
    [8.17e11, 1.69e-6],
    [1.19e12, 2.42e-7],
    [2.15e12, 1.13e-8],
    [2.90e12, 2.53e-9],
    [3.36e12, 1.20e-9],
    [1.11e13, 1.59e-10],
    [2.70e13, 1.09e-10],
    [1.74e14, 2.68e-10],
    [1.88e15, 2.72e-9],
    [2.56e17, 3.02e-7],
    [3.47e18, 4.81e-6],
    [3.01e19, 3.62e-5],
    [1.44e20, 1.87e-4],
    [9.23e20, 1.13e-3],
    [2.63e22, 3.51e-2],
    [3.31e23, 4.81e-1],
])

acharya_tau = ACHARYA_CMB_DATA[:, 0]
acharya_f = ACHARYA_CMB_DATA[:, 1]
acharya_interp = interp1d(
    np.log10(acharya_tau), np.log10(acharya_f),
    kind='cubic', bounds_error=False, fill_value='extrapolate'
)

def get_cmb_constraint(tau_X):
    tau_cutoff = acharya_tau[0]
    if tau_X < tau_cutoff:
        return np.inf
    log_tau = np.log10(tau_X)
    tau_min = acharya_tau[0]
    tau_max = acharya_tau[-1]
    if tau_X > tau_max:
        log_f = np.log10(acharya_f[-1]) + (log_tau - np.log10(tau_max))
    else:
        log_f = acharya_interp(log_tau)
    return min(10**log_f, 1.0)

print(f"Loaded {len(ACHARYA_CMB_DATA)} Acharya+2019 data points")
print(f"  tau range: {acharya_tau[0]:.2e} – {acharya_tau[-1]:.2e} s")
print(f"  f range:   {acharya_f[0]:.2e} – {acharya_f[-1]:.2e}")


## PBH Evaporation with Memory Burden

The transition function $h(M)$ smoothly interpolates between semi-classical evaporation and the memory-burden suppressed regime:

$$h(M) = \frac{1}{2}\left[1 + \tanh\left(\frac{M - qM_i}{\delta q M_i / 2}\right)\right]$$

The evaporation rate is:

$$\frac{dM}{dt} = -\exp\left[h(M) \ln r_{SC} + (1-h(M)) \ln r_{MB}\right]$$


In [ ]:
# ==================== PBH Evaporation ====================
class PBHEvaporation:
    def __init__(self, M_i, q=0.5, delta=0.1, k=2):
        self.M_i = M_i
        self.q = q
        self.delta = delta
        self.k = k
        self.M_trans = q * M_i
        self.S_trans = S_ref * (self.M_trans / M_ref)**2
        rate_SC_at_trans = evap_coef / self.M_trans**2
        self.rate_MB = rate_SC_at_trans / (self.S_trans ** k)
        self.t_trans = (M_i**3 - self.M_trans**3) / (3 * evap_coef)

    def transition_function(self, M):
        if self.delta == 0:
            return 1.0 if M >= self.M_trans else 0.0
        width = self.delta * self.M_trans / 2.0
        arg = (M - self.M_trans) / width
        return 0.5 * (1.0 + np.tanh(arg))

    def evaporation_rate(self, M):
        if M <= 1e-100:
            return 0
        h = self.transition_function(M)
        rate_SC = evap_coef / M**2
        rate_MB = self.rate_MB
        if rate_SC > 0 and rate_MB > 0:
            log_rate = h * np.log(rate_SC) + (1 - h) * np.log(rate_MB)
            return -np.exp(log_rate)
        return 0

    def compute_evolution(self, n_points=5000):
        t_min = 1e-40
        t_CMB_relevant = 1e18
        if self.delta == 0:
            t_MB_evap = self.M_trans / self.rate_MB if self.rate_MB > 0 else 1e100
            t_max = min(max(10 * self.t_trans, t_CMB_relevant),
                       self.t_trans + 10 * t_MB_evap)
        else:
            t_MB_evap = self.M_trans / self.rate_MB if self.rate_MB > 0 else 1e100
            t_max = min(max(100 * self.t_trans, t_CMB_relevant),
                       self.t_trans + 100 * t_MB_evap)
        t_max = max(t_max, 1e-20)
        t_max = min(t_max, 1e45)
        t_array = np.logspace(np.log10(t_min), np.log10(t_max), n_points)

        def dM_dt(t, M):
            return self.evaporation_rate(M)

        try:
            sol = solve_ivp(dM_dt, [t_min, t_max], [self.M_i],
                            t_eval=t_array, method='RK45', rtol=1e-8, atol=1e-12)
            M_array = np.maximum(sol.y[0], 0)
        except:
            M_array = np.full_like(t_array, self.M_i)
            mask_SC = t_array < self.t_trans
            M_array[mask_SC] = np.maximum(
                0, (self.M_i**3 - 3 * evap_coef * t_array[mask_SC])**(1/3))
            mask_MB = t_array >= self.t_trans
            M_array[mask_MB] = np.maximum(
                0, self.M_trans - self.rate_MB * (t_array[mask_MB] - self.t_trans))

        dMdt_array = np.array([self.evaporation_rate(M) for M in M_array])
        return t_array, M_array, dMdt_array

print("PBHEvaporation class defined.")


## Three-Phase CMB Constraint

The CMB constraint is computed from **three distinct phases** of PBH evaporation:

1. **SC phase** ($M > qM_i$): Standard semi-classical evaporation — fast, follows Page's formula.
2. **Transition phase** ($M \approx qM_i$): Smooth crossover region where memory burden begins to take effect.
3. **MB phase** ($M < qM_i$): Memory-burden dominated — evaporation is exponentially suppressed by $S_{\rm Page}^{-k}$.

The final constraint is the **most restrictive** of the three:

$$f_{PBH}^{CMB} = \min(f_{SC}, f_{Trans}, f_{MB})$$


In [ ]:
# ==================== Three-Phase CMB Constraints ====================

def compute_SC_constraint(t_array, M_array, dMdt_array, M_i, q):
    mask_SC = M_array > q * M_i
    if not np.any(mask_SC):
        return np.inf, None
    t_SC = t_array[mask_SC]
    dMdt_SC = dMdt_array[mask_SC]
    dEdt_SC = -dMdt_SC * C_LIGHT**2
    dt = np.diff(t_SC)
    if len(dt) == 0:
        return np.inf, None
    dEdt_mid = 0.5 * (dEdt_SC[:-1] + dEdt_SC[1:])
    E_cumulative = np.cumsum(dEdt_mid * dt)
    E_total = E_cumulative[-1]
    if E_total <= 0:
        return np.inf, None
    idx_median = np.argmin(np.abs(E_cumulative - 0.5 * E_total))
    t_median = 0.5 * (t_SC[:-1][idx_median] + t_SC[1:][idx_median])
    tau_X = t_median
    f_X = get_cmb_constraint(tau_X)
    if np.isinf(f_X):
        return np.inf, tau_X
    idx_t = np.argmin(np.abs(t_array - t_median))
    dMdt_at_median = abs(dMdt_array[idx_t])
    if dMdt_at_median <= 0:
        return np.inf, tau_X
    f_PBH_SC = f_X * M_i / (np.e * tau_X * dMdt_at_median)
    return f_PBH_SC, tau_X


def compute_MB_constraint(t_array, M_array, dMdt_array, M_i, q, k, delta):
    if delta == 0:
        return np.inf, None
    idx_trans = np.argmin(np.abs(M_array - q * M_i))
    t_MB = t_array[idx_trans]
    mask_positive = M_array > 1e-30
    t_evap_end = t_array[mask_positive][-1] if np.any(mask_positive) else t_array[-1]
    rate_MB = abs(dMdt_array[idx_trans]) if idx_trans < len(dMdt_array) else 0
    if rate_MB > 0 and M_array[idx_trans] > 0:
        t_evap_true = t_MB + M_array[idx_trans] / rate_MB
        t_evap_end = min(t_evap_end, t_evap_true)
    tau_min = max(10 * t_MB, 1e9)
    tau_max = max(1e-3 * t_evap_end, 10 * tau_min)
    if tau_min >= tau_max:
        return np.inf, None
    tau_values = np.logspace(np.log10(tau_min), np.log10(tau_max), 200)
    f_PBH_min = np.inf
    best_tau = None
    rho_DM = Omega_DM * rho_c
    for tau_X in tau_values:
        f_X = get_cmb_constraint(tau_X)
        if np.isinf(f_X) or f_X <= 0:
            continue
        t1 = max(1e-3 * tau_X, t_array[0])
        t2 = min(1e2 * tau_X, t_array[-1])
        mask_window = (t_array >= t1) & (t_array <= t2) & (M_array > 0)
        if not np.any(mask_window):
            continue
        dEdt = -dMdt_array[mask_window] * C_LIGHT**2
        t_win = t_array[mask_window]
        if len(t_win) < 2:
            continue
        E_PBH_total = np.trapezoid(dEdt, t_win)
        if E_PBH_total <= 0:
            continue
        exp_factor = np.exp(-t1/tau_X) - np.exp(-t2/tau_X)
        E_X_max = rho_DM * C_LIGHT**2 * exp_factor
        f_PBH = f_X * E_X_max * M_i / (rho_DM * E_PBH_total)
        if f_PBH < f_PBH_min:
            f_PBH_min = f_PBH
            best_tau = tau_X
    return f_PBH_min, best_tau


def compute_transition_constraint(t_array, M_array, dMdt_array, M_i, q, delta):
    if delta <= 0 or delta > 0.5:
        return np.inf, None
    M_center = q * M_i
    M_low = max(M_center * (1 - 5*delta), 1e-100)
    M_high = min(M_center * (1 + 5*delta), M_i * 0.999)
    mask_trans = (M_array >= M_low) & (M_array <= M_high) & (M_array > 0)
    if not np.any(mask_trans):
        return np.inf, None
    n_segments = 10
    seg_indices_all = np.where(mask_trans)[0]
    seg_size = max(len(seg_indices_all) // n_segments, 1)
    idx_segments = [seg_indices_all[i:i+seg_size] for i in range(0, len(seg_indices_all), seg_size)]
    f_PBH_min = np.inf
    best_tau = None
    rho_DM = Omega_DM * rho_c
    for seg_indices in idx_segments:
        if len(seg_indices) < 2:
            continue
        t_seg = t_array[seg_indices]
        dMdt_seg = dMdt_array[seg_indices]
        dEdt_seg = -dMdt_seg * C_LIGHT**2
        if np.all(dEdt_seg <= 0):
            continue
        E_seg = np.trapezoid(dEdt_seg, t_seg)
        if E_seg <= 0:
            continue
        tau_seg = np.trapezoid(t_seg * dEdt_seg, t_seg) / E_seg
        dMdt_avg = np.mean(np.abs(dMdt_seg))
        if dMdt_avg <= 0 or tau_seg <= 0:
            continue
        f_X = get_cmb_constraint(tau_seg)
        if np.isinf(f_X):
            continue
        f_PBH_seg = f_X * M_i / (np.e * tau_seg * dMdt_avg)
        t1 = max(1e-3 * tau_seg, t_array[0])
        t2 = min(1e2 * tau_seg, t_array[-1])
        mask_window = (t_array >= t1) & (t_array <= t2) & (M_array > 0)
        if np.any(mask_window):
            dEdt_win = -dMdt_array[mask_window] * C_LIGHT**2
            t_win = t_array[mask_window]
            if len(t_win) >= 2:
                E_PBH = np.trapezoid(dEdt_win, t_win)
                exp_factor = np.exp(-t1/tau_seg) - np.exp(-t2/tau_seg)
                E_X = rho_DM * C_LIGHT**2 * exp_factor
                if E_PBH > 0:
                    f_PBH_energy = f_X * E_X * M_i / (rho_DM * E_PBH)
                    f_PBH_seg = min(f_PBH_seg, f_PBH_energy)
        if f_PBH_seg < f_PBH_min:
            f_PBH_min = f_PBH_seg
            best_tau = tau_seg
    return f_PBH_min, best_tau


def compute_f_PBH_limit(M_i, q=0.5, delta=0.1, k=2):
    pbh = PBHEvaporation(M_i, q, delta, k)
    t_array, M_array, dMdt_array = pbh.compute_evolution()
    f_SC, tau_SC = compute_SC_constraint(t_array, M_array, dMdt_array, M_i, q)
    f_trans, tau_trans = compute_transition_constraint(t_array, M_array, dMdt_array, M_i, q, delta)
    f_MB, tau_MB = compute_MB_constraint(t_array, M_array, dMdt_array, M_i, q, k, delta)
    f_PBH = min(f_SC, f_trans, f_MB)
    if f_SC <= f_trans and f_SC <= f_MB:
        phase = 'SC'
        tau = tau_SC
    elif f_trans <= f_MB:
        phase = 'Trans'
        tau = tau_trans
    else:
        phase = 'MB'
        tau = tau_MB
    if np.isinf(f_PBH):
        f_PBH = 1.0
    return min(max(f_PBH, 1e-14), 1.0), phase, tau

print("Three-phase CMB constraint functions defined.")
print("  - compute_SC_constraint(): SC-phase constraint")
print("  - compute_transition_constraint(): Transition-phase constraint")
print("  - compute_MB_constraint(): MB-phase constraint")
print("  - compute_f_PBH_limit(): Combined CMB limit (min of three phases)")


## BBN Constraint

The BBN constraint uses the **decaying-particle mapping** method from Keith *et al.* 2020 [29] and Kawasaki *et al.* 2018 [37]:

1. Compute the PBH evaporation history.
2. Identify median injection time for photons ($t > 10^4$ s) and hadrons ($t < 10^4$ s).
3. Map these times onto equivalent decaying-particle lifetimes.
4. Compute effective $m_X Y_X$ from the total energy injected during BBN.
5. Compare with the BBN limits from [37].

Additionally, for the memory-burden phase we scan lifetimes in $[10 t_{MB}, 10^{-3} t_{evap}]$ as described in Supplemental Material S1.


In [ ]:
# ==================== BBN Constraint ====================

# BBN epoch boundaries
t_BBN_START = 1.0      # [s] BBN begins (n/p freeze-out)
t_BBN_END = 1e6        # [s] BBN effectively ends
t_SPLIT_S = 1e4        # [s] Boundary between hadro- and photo-disintegration

# Kawasaki+2018 BBN limit curve for electromagnetic decays
tau_nodes = np.logspace(0, 14, 100)
log_tau = np.log10(tau_nodes)
log_limit = -19.0 + 0.5 * (log_tau - 5.0)**2 / 2.0
log_limit += np.where(log_tau < 3.0, 1.0 * (3.0 - log_tau), 0.0)
log_limit = np.where(
    log_tau > 7.5,
    -19.0 + 0.5 * (7.5 - 5.0)**2 / 2.0 + (log_tau - 7.5),
    log_limit
)
_bbn_limit_interp = interp1d(log_tau, log_limit, kind='cubic', fill_value='extrapolate')

def bbn_limit(tau):
    return 10.0 ** _bbn_limit_interp(np.log10(tau))

RHO_OVER_S_GEV = 1e-3   # [GeV] ~ rho_DM / s


def get_evolution(M_i, q=0.5, delta=0.1, k=2, n_points=5000):
    M_trans = q * M_i
    S_trans = S_ref * (M_trans / M_ref) ** 2
    rate_SC_at_trans = evap_coef / M_trans ** 2
    rate_MB = rate_SC_at_trans / (S_trans ** k)
    t_trans = (M_i**3 - M_trans**3) / (3.0 * evap_coef)

    def trans_fn(M):
        if delta == 0:
            return 1.0 if M >= M_trans else 0.0
        width = delta * M_trans / 2.0
        return 0.5 * (1.0 + np.tanh((M - M_trans) / width))

    def dM_dt(t, M):
        if M <= 1e-100:
            return 0.0
        h = trans_fn(M)
        rate_SC = evap_coef / M**2
        log_rate = h * np.log(rate_SC) + (1.0 - h) * np.log(rate_MB)
        return -np.exp(log_rate)

    t_min = 1e-40
    t_CMB = 1e18
    t_MB_evap = M_trans / rate_MB if rate_MB > 0 else 1e100

    if delta == 0:
        t_max = min(max(10.0 * t_trans, t_CMB), t_trans + 10.0 * t_MB_evap)
    else:
        t_max = min(max(100.0 * t_trans, t_CMB),
                    t_trans + 100.0 * t_MB_evap)

    t_max = max(t_max, t_min * 10.0)
    t_max = min(t_max, 1e45)

    t_eval = np.logspace(np.log10(t_min), np.log10(t_max), n_points)
    t_eval = np.clip(t_eval, t_min, t_max)

    sol = solve_ivp(dM_dt, [t_min, t_max], [M_i],
                    t_eval=t_eval, method='RK45',
                    rtol=1e-8, atol=1e-12)
    return sol.t, np.maximum(sol.y[0], 0.0)


def compute_bbn_constraint(M_i, q=0.5, delta=0.1, k=2):
    t_arr, M_arr = get_evolution(M_i, q, delta, k)

    # If PBH evaporated before BBN, no BBN constraint
    if t_arr[-1] < t_BBN_START:
        return 1.0

    M_start = np.interp(t_BBN_START, t_arr, M_arr,
                        left=M_arr[0], right=M_arr[-1])
    M_end = np.interp(t_BBN_END, t_arr, M_arr,
                      left=M_arr[0], right=M_arr[-1])
    Delta_M = M_start - M_end
    if Delta_M <= 1e-100:
        return 1.0

    frac = Delta_M / M_i
    mask_BBN = (t_arr >= t_BBN_START) & (t_arr <= min(t_BBN_END, t_arr[-1]))
    if not np.any(mask_BBN):
        return 1.0

    t_bbn = t_arr[mask_BBN]
    M_bbn = M_arr[mask_BBN]

    dMdt = np.gradient(M_bbn, t_bbn)
    energy_rate = -dMdt * C_LIGHT**2
    energy_rate = np.maximum(energy_rate, 0.0)
    dt = np.gradient(t_bbn)
    cum_energy = np.cumsum(energy_rate * dt)
    total_energy = cum_energy[-1]
    if total_energy <= 0.0:
        return 1.0

    results = []

    # 1) Photodisintegration era (t > 1e4 s)
    mask_phot = t_bbn > t_SPLIT_S
    if np.any(mask_phot):
        E_phot = energy_rate[mask_phot] * dt[mask_phot]
        cum_E = np.cumsum(E_phot)
        idx_med = np.argmin(np.abs(cum_E - 0.5 * cum_E[-1]))
        t_med_phot = t_bbn[mask_phot][idx_med]
        tau_phot = t_med_phot / 0.79
        limit_phot = bbn_limit(tau_phot)
        mY_eff_phot = frac * RHO_OVER_S_GEV
        f_max_phot = limit_phot / mY_eff_phot if mY_eff_phot > 0 else 1.0
        results.append(f_max_phot)

    # 2) Hadrodisintegration era (t < 1e4 s)
    mask_had = t_bbn <= t_SPLIT_S
    if np.any(mask_had):
        E_had = energy_rate[mask_had] * dt[mask_had]
        cum_E = np.cumsum(E_had)
        idx_med = np.argmin(np.abs(cum_E - 0.5 * cum_E[-1]))
        t_med_had = t_bbn[mask_had][idx_med]
        tau_had = t_med_had / 0.71
        limit_had = bbn_limit(tau_had)
        mY_eff_had = frac * RHO_OVER_S_GEV
        f_max_had = limit_had / mY_eff_had if mY_eff_had > 0 else 1.0
        results.append(f_max_had)

    # 3) Memory-burden phase scan (main paper S1)
    t_MB = (M_i**3 - (q * M_i)**3) / (3.0 * evap_coef)
    t_evap_total = t_arr[-1]
    tau_MB_min = max(10.0 * t_MB,
                     t_bbn[0] if len(t_bbn) > 0 else t_MB)
    tau_MB_max = 1e-3 * t_evap_total

    if delta > 0 and tau_MB_min < tau_MB_max:
        tau_vals = np.logspace(np.log10(tau_MB_min),
                               np.log10(tau_MB_max), 20)
        for tau in tau_vals:
            t_win_start = 1e-3 * tau
            t_win_end = min(1e2 * tau, t_arr[-1])
            mask_win = (t_bbn >= t_win_start) & (t_bbn <= t_win_end)
            if not np.any(mask_win):
                continue

            E_win = np.sum(energy_rate[mask_win] * dt[mask_win])
            if E_win <= 0:
                continue

            M_s = np.interp(t_win_start, t_bbn, M_bbn,
                            left=M_bbn[0], right=M_bbn[-1])
            M_e = np.interp(t_win_end, t_bbn, M_bbn,
                            left=M_bbn[0], right=M_bbn[-1])
            Delta_M_win = max(0.0, M_s - M_e)
            frac_win = Delta_M_win / M_i

            mY_eff_win = frac_win * RHO_OVER_S_GEV
            limit_win = bbn_limit(tau)
            f_max_win = limit_win / mY_eff_win if mY_eff_win > 0 else 1.0
            results.append(f_max_win)

    if len(results) == 0:
        return 1.0

    f_bbn = float(np.min(results))
    f_bbn = min(1.0, max(1e-30, f_bbn))
    return f_bbn

print("BBN constraint functions defined.")
print("  - bbn_limit(tau): Kawasaki+2018 BBN limit curve")
print("  - compute_bbn_constraint(): Full BBN constraint with MB phase scan")


## Test CMB Constraints

Compute CMB constraints for a range of PBH masses and both transition types.


In [ ]:
# ==================== Test CMB Constraints ====================
test_masses = [1e4, 1e5, 1e6, 1e7, 1e8, 1e9, 1e10, 1e12, 1e14, 1e16]

print("=" * 70)
print("CMB CONSTRAINT TEST")
print("=" * 70)

for delta, label in [(0.0, "Step-like (delta=0)"), (0.1, "Continuous (delta=0.1)")]:
    print(f"
Parameters: q=0.5, {label}")
    print(f"{'M_i (g)':<12} {'f_PBH,0':<14} {'Phase':<8} {'tau (s)':<14} {'Status':<15}")
    print("-" * 65)
    for M in test_masses:
        f_lim, phase, tau = compute_f_PBH_limit(M, 0.5, delta, 2)
        status = "Allowed as DM" if f_lim >= 1.0 else "CMB Constrained"
        tau_str = f"{tau:.2e}" if tau else "N/A"
        print(f"{M:<12.0e} {f_lim:<14.2e} {phase:<8} {tau_str:<14} {status:<15}")


### CMB Constraint Curves


In [ ]:
# ==================== CMB Constraint Plot ====================
M_grid = np.logspace(4, 16.5, 100)
fig, ax = plt.subplots(figsize=(10, 7))

params = [
    (0.5, 0.0, 'Step-like (delta=0, q=0.5)', 'red', '--', 3),
    (0.5, 0.1, 'Continuous (delta=0.1, q=0.5)', 'blue', '-', 2.5),
    (0.2, 0.1, 'Continuous (delta=0.1, q=0.2)', 'green', ':', 2.5),
    (0.8, 0.1, 'Continuous (delta=0.1, q=0.8)', 'magenta', '-.', 2.5),
]

for q, delta, label, color, style, lw in params:
    print(f"Computing: {label}...")
    f_limits = []
    for M in M_grid:
        try:
            f_lim, _, _ = compute_f_PBH_limit(M, q, delta, 2)
            f_limits.append(f_lim)
        except:
            f_limits.append(1e-14)
    ax.loglog(M_grid, f_limits, color=color, linestyle=style,
              linewidth=lw, label=label)

M_evap = 5e14
ax.axvline(M_evap, color='gray', linestyle='-', alpha=0.5, linewidth=1)
ax.fill_between([1e4, M_evap], [1e-14, 1e-14], [2, 2],
                alpha=0.15, color='gray', label='Evaporated (std)')
ax.axhline(1.0, color='gray', linestyle='-', alpha=0.3, linewidth=1)
ax.text(2e4, 1.3, '$f_{PBH,0}=1$', fontsize=11, color='gray')
ax.axvline(4e16, color='purple', linestyle='-.', alpha=0.5, linewidth=1.5)
ax.text(5e16, 0.3, '$M \sim 4\times 10^{16}$ g\nthreshold', fontsize=10,
        bbox=dict(boxstyle='round', facecolor='lavender', alpha=0.7))

ax.set_xlabel('Initial PBH Mass $M_i$ [g]', fontsize=14)
ax.set_ylabel('$f_{PBH,0}$ Upper Limit', fontsize=14)
ax.set_title('CMB Constraints on PBHs with Memory Burden Effect', fontsize=15)
ax.legend(loc='lower right', fontsize=11, framealpha=0.9)
ax.grid(True, alpha=0.3, which='both')
ax.set_xlim(1e4, 1e17)
ax.set_ylim(1e-14, 2)
plt.tight_layout()
plt.show()


## Test BBN Constraints


In [ ]:
# ==================== Test BBN Constraints ====================
print("=" * 70)
print("BBN CONSTRAINT TEST")
print("=" * 70)

test_cases = [1e4, 1e5, 1e6, 1e7, 1e8, 1e9, 1e10, 1e12, 1e14]

print("
--- delta=0.1 (continuous-like) ---")
for M_i in test_cases:
    f = compute_bbn_constraint(M_i, 0.5, 0.1, 2)
    print(f"  M_i={M_i:.2e}  f_BBN={f:.2e}")

print("
--- delta=0 (step-like) ---")
for M_i in test_cases:
    f = compute_bbn_constraint(M_i, 0.5, 0.0, 2)
    print(f"  M_i={M_i:.2e}  f_BBN={f:.2e}")


### BBN Constraint Curves


In [ ]:
# ==================== BBN Constraint Plot ====================
M_vals = np.logspace(4, 14, 100)
f_bbn_01 = [compute_bbn_constraint(M, 0.5, 0.1, 2) for M in M_vals]
f_bbn_00 = [compute_bbn_constraint(M, 0.5, 0.0, 2) for M in M_vals]

plt.figure(figsize=(8, 6))
plt.loglog(M_vals, f_bbn_01, 'g-', lw=2, label='BBN  delta=0.1, q=0.5')
plt.loglog(M_vals, f_bbn_00, 'g--', lw=2, label='BBN  delta=0, q=0.5')
plt.xlabel(r'$M_i$ [g]', fontsize=13)
plt.ylabel(r'$f_{\rm PBH}$', fontsize=13)
plt.title('BBN Constraint for Memory-Burden PBHs', fontsize=14)
plt.legend(fontsize=11)
plt.xlim(1e4, 1e14)
plt.ylim(1e-20, 2.0)
plt.grid(True, which='both', ls='--', alpha=0.5)
plt.tight_layout()
plt.show()


## Combined CMB + BBN Constraints

The combined constraint at each mass is the **more restrictive** of the two:

$$f_{\rm combined}(M_i) = \min(f_{CMB}(M_i), f_{BBN}(M_i))$$


In [ ]:
# ==================== Combined CMB + BBN ====================

def is_evaporated_by_now(M_i, q=0.5, delta=0.1, k=2, t_age_s=4.3e17):
    t_evap_std = M_i**3 / (3.0 * evap_coef)
    return t_evap_std < t_age_s


def compute_combined_constraint(M_i, q=0.5, delta=0.1, k=2):
    f_cmb, _, _ = compute_f_PBH_limit(M_i, q, delta, k)
    f_bbn = compute_bbn_constraint(M_i, q, delta, k)
    return min(f_cmb, f_bbn)


M_vals = np.logspace(4, 16, 50)
q = 0.5
k = 2

f_cmb_01 = []
f_cmb_00 = []
f_bbn_01 = []
f_comb_01 = []
evaporated = []

print("Computing combined constraints for Fig. 3 reproduction ...")
for i, M in enumerate(M_vals):
    if i % 10 == 0:
        print(f"  {i}/50  M={M:.2e}")

    f_cmb_01.append(compute_f_PBH_limit(M, q, 0.1, k)[0])
    f_cmb_00.append(compute_f_PBH_limit(M, q, 0.0, k)[0])
    f_bbn_01.append(compute_bbn_constraint(M, q, 0.1, k))
    f_comb_01.append(min(f_cmb_01[-1], f_bbn_01[-1]))
    evaporated.append(is_evaporated_by_now(M, q, 0.1, k))

f_cmb_01 = np.array(f_cmb_01)
f_cmb_00 = np.array(f_cmb_00)
f_bbn_01 = np.array(f_bbn_01)
f_comb_01 = np.array(f_comb_01)
evaporated = np.array(evaporated)

if np.any(evaporated):
    M_evap_max = np.max(M_vals[evaporated])
else:
    M_evap_max = 0.0

print("Done!")


### Fig. 3 Reproduction

Main figure: CMB (continuous & step-like), BBN, and combined constraints.


In [ ]:
# ==================== Fig. 3 Reproduction ====================
fig, ax = plt.subplots(figsize=(9, 7))

# Evaporated region (grey)
if M_evap_max > 0:
    ax.axvspan(M_vals[0], M_evap_max, color='grey', alpha=0.25,
               label='Evaporated by now')

# CMB curves
ax.loglog(M_vals, f_cmb_01, 'b-', lw=2.5,
          label=r'CMB continuous ($\delta=0.1$)')
ax.loglog(M_vals, f_cmb_00, 'k--', lw=2.5,
          label=r'CMB step-like ($\delta=0$)')

# BBN curve
ax.loglog(M_vals, f_bbn_01, 'g-', lw=2.0,
          label=r'BBN ($\delta=0.1$)')

# Combined curve
ax.loglog(M_vals, f_comb_01, 'r-', lw=3.0,
          label=r'Combined CMB+BBN ($\delta=0.1$)')

# Formatting
ax.set_xlabel(r'Initial Mass  $M_i$ [g]', fontsize=14)
ax.set_ylabel(r'$f_{\rm PBH,0}$', fontsize=14)
ax.set_title(
    'PBH Constraints: CMB + BBN (Memory Burden, q=0.5, k=2)',
    fontsize=15)
ax.set_xlim(1e4, 1e16)
ax.set_ylim(1e-20, 2.0)
ax.legend(fontsize=11, loc='lower left')
ax.grid(True, which='both', ls='--', alpha=0.4)

if M_evap_max > 0:
    ax.text(M_evap_max * 0.3, 1.5, 'Evaporated\nby now',
            fontsize=12, color='grey', ha='center', va='top')

plt.tight_layout()
plt.show()


### Parameter Scan: Varying delta


In [ ]:
# ==================== Delta Scan ====================
M_vals = np.logspace(4, 16, 50)
q = 0.5
k = 2
deltas = [0.0, 0.1, 0.3]
colors = ['black', 'blue', 'purple']

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax_idx, ax in enumerate(axes):
    for delta, color in zip(deltas, colors):
        f_cmb = []
        f_bbn = []
        f_comb = []
        for M in M_vals:
            fc = compute_f_PBH_limit(M, q, delta, k)[0]
            fb = compute_bbn_constraint(M, q, delta, k)
            f_cmb.append(fc)
            f_bbn.append(fb)
            f_comb.append(min(fc, fb))

        if ax_idx == 0:
            ax.loglog(M_vals, f_cmb, color=color, lw=2.5,
                      label=f'CMB  delta={delta}')
            ax.loglog(M_vals, f_bbn, color=color, lw=1.5, ls='--',
                      label=f'BBN  delta={delta}')
            ax.set_title('CMB and BBN separately', fontsize=14)
        else:
            ax.loglog(M_vals, f_comb, color=color, lw=2.5,
                      label=f'Combined  delta={delta}')
            ax.set_title('Combined CMB+BBN', fontsize=14)

    ax.set_xlabel(r'$M_i$ [g]', fontsize=13)
    ax.set_ylabel(r'$f_{\rm PBH,0}$', fontsize=13)
    ax.set_xlim(1e4, 1e16)
    ax.set_ylim(1e-20, 2.0)
    ax.legend(fontsize=10, loc='lower left')
    ax.grid(True, which='both', ls='--', alpha=0.4)

plt.tight_layout()
plt.show()


### Parameter Scan: Varying q


In [ ]:
# ==================== q Scan ====================
M_vals = np.logspace(4, 16, 50)
delta = 0.1
k = 2
qs = [0.2, 0.5, 0.8]
colors = ['red', 'blue', 'green']

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax_idx, ax in enumerate(axes):
    for q, color in zip(qs, colors):
        f_cmb = []
        f_bbn = []
        f_comb = []
        for M in M_vals:
            fc = compute_f_PBH_limit(M, q, delta, k)[0]
            fb = compute_bbn_constraint(M, q, delta, k)
            f_cmb.append(fc)
            f_bbn.append(fb)
            f_comb.append(min(fc, fb))

        if ax_idx == 0:
            ax.loglog(M_vals, f_cmb, color=color, lw=2.5,
                      label=f'CMB  q={q}')
            ax.loglog(M_vals, f_bbn, color=color, lw=1.5, ls='--',
                      label=f'BBN  q={q}')
            ax.set_title('CMB and BBN separately', fontsize=14)
        else:
            ax.loglog(M_vals, f_comb, color=color, lw=2.5,
                      label=f'Combined  q={q}')
            ax.set_title('Combined CMB+BBN', fontsize=14)

    ax.set_xlabel(r'$M_i$ [g]', fontsize=13)
    ax.set_ylabel(r'$f_{\rm PBH,0}$', fontsize=13)
    ax.set_xlim(1e4, 1e16)
    ax.set_ylim(1e-20, 2.0)
    ax.legend(fontsize=10, loc='lower left')
    ax.grid(True, which='both', ls='--', alpha=0.4)

plt.tight_layout()
plt.show()


## Four Transition Functions Validation

Validation of four smooth transition functions proposed in the companion paper "Several possible PBHs transition continuously from the semiclassical phase to the memory burden effect phase":

- **h1**: Logistic ($\tanh$)
- **h2**: Error function ($\mathrm{erf}$)
- **h3**: Arctangent ($\arctan$)
- **h4**: Fractional radical

Each function is tested with **both** distributions:
- **Additive**: $dM/dt = h(M) \cdot \mathrm{rate}_{SC} + (1-h(M)) \cdot \mathrm{rate}_{MB}$
- **Multiplicative**: $dM/dt = \mathrm{rate}_{SC}^{h(M)} \cdot \mathrm{rate}_{MB}^{(1-h(M))}$


In [ ]:
# ==================== Four Transition Functions ====================

def h1_tanh(M, M_i, q, delta):
    M_trans = q * M_i
    width = delta * M_trans / 2.0
    if width < 1e-50:
        return 1.0 if M >= M_trans else 0.0
    return 0.5 * (1.0 + np.tanh((M - M_trans) / width))


def h2_erf(M, M_i, q, delta):
    M_trans = q * M_i
    width = delta * M_trans / 2.0
    if width < 1e-50:
        return 1.0 if M >= M_trans else 0.0
    return 0.5 * (1.0 + erf((M - M_trans) / width))


def h3_arctan(M, M_i, q, delta):
    M_trans = q * M_i
    width = delta * M_trans
    if width < 1e-50:
        return 1.0 if M >= M_trans else 0.0
    return 0.5 + (1.0 / np.pi) * np.arctan((M - M_trans) / width)


def h4_fractional_radical(M, M_i, q, delta):
    M_trans = q * M_i
    denom = delta * M_trans
    if denom < 1e-50:
        return 1.0 if M >= M_trans else 0.0
    x = (M - M_trans) / denom
    return 0.5 + x / (2.0 * np.sqrt(x**2 + delta**2))


TRANSITION_FUNCTIONS = {
    'h1_tanh': h1_tanh,
    'h2_erf': h2_erf,
    'h3_arctan': h3_arctan,
    'h4_radical': h4_fractional_radical,
}

DISTRIBUTIONS = ['additive', 'multiplicative']

print("Four transition functions defined:")
for name in TRANSITION_FUNCTIONS:
    print(f"  - {name}")
print(f"Two distributions: {DISTRIBUTIONS}")


In [ ]:
# ==================== Custom Evolution & Constraints ====================

def get_custom_evolution(M_i, q, delta, k, h_func, dist_type, n_points=5000):
    M_trans = q * M_i
    S_trans = S_ref * (M_trans / M_ref)**2
    rate_SC_at_trans = evap_coef / M_trans**2
    rate_MB = rate_SC_at_trans / (S_trans ** k)
    t_trans = (M_i**3 - M_trans**3) / (3.0 * evap_coef)

    def dM_dt(t, M):
        if M <= 1e-100:
            return 0.0
        h = h_func(M, M_i, q, delta)
        rate_SC = evap_coef / M**2
        if dist_type == 'additive':
            rate = h * rate_SC + (1.0 - h) * rate_MB
        else:
            log_rate = h * np.log(rate_SC) + (1.0 - h) * np.log(rate_MB)
            rate = np.exp(log_rate)
        return -rate

    t_min = 1e-40
    t_CMB = 1e18
    t_MB_evap = M_trans / rate_MB if rate_MB > 0 else 1e100

    if delta == 0:
        t_max = min(max(10.0 * t_trans, t_CMB), t_trans + 10.0 * t_MB_evap)
    else:
        t_max = min(max(100.0 * t_trans, t_CMB),
                    t_trans + 100.0 * t_MB_evap)

    t_max = max(t_max, t_min * 10.0)
    t_max = min(t_max, 1e45)

    t_eval = np.logspace(np.log10(t_min), np.log10(t_max), n_points)
    sol = solve_ivp(dM_dt, [t_min, t_max], [M_i],
                    t_eval=t_eval, method='RK45',
                    rtol=1e-8, atol=1e-12)
    return sol.t, np.maximum(sol.y[0], 0.0)


def get_custom_cmb(M_i, q, delta, k, h_func, dist_type):
    M_trans = q * M_i
    S_trans = S_ref * (M_trans / M_ref)**2
    rate_SC_at_trans = evap_coef / M_trans**2
    rate_MB = rate_SC_at_trans / (S_trans ** k)

    h = h_func(M_i, M_i, q, delta)
    rate_SC = evap_coef / M_i**2

    if dist_type == 'additive':
        rate_eff = h * rate_SC + (1.0 - h) * rate_MB
    else:
        log_rate = h * np.log(rate_SC) + (1.0 - h) * np.log(rate_MB)
        rate_eff = np.exp(log_rate)
    Lambda_max = M_i / rate_eff

    t_arr, M_arr = get_custom_evolution(M_i, q, delta, k, h_func, dist_type)

    M_end = M_arr[-1]
    if M_end <= 0:
        M_end = 0
    M_evap = M_i - M_end
    energy_ratio = M_evap / M_i

    rho_DM_ratio = Omega_DM * rho_c / rho_DM_ref
    f_ratio = np.sqrt(M_ref / M_i) * rho_DM_ratio / Lambda_CMB
    best_fit_f = f_ratio * np.sqrt(energy_ratio)
    return best_fit_f * 1e3


def get_custom_bbn(M_i, q, delta, k, h_func, dist_type):
    RHO_OVER_S_GEV = 1e-3
    t_BBN_START = 1.0
    t_BBN_END = 1e6
    t_SPLIT_S = 1e4

    t_arr, M_arr = get_custom_evolution(M_i, q, delta, k, h_func, dist_type)

    if t_arr[-1] < t_BBN_START:
        return 1.0

    M_start = np.interp(t_BBN_START, t_arr, M_arr,
                        left=M_arr[0], right=M_arr[-1])
    M_end = np.interp(t_BBN_END, t_arr, M_arr,
                      left=M_arr[0], right=M_arr[-1])
    Delta_M = M_start - M_end
    if Delta_M <= 1e-100:
        return 1.0

    frac = Delta_M / M_i
    mask_BBN = (t_arr >= t_BBN_START) & (t_arr <= min(t_BBN_END, t_arr[-1]))
    if not np.any(mask_BBN):
        return 1.0

    t_bbn = t_arr[mask_BBN]
    M_bbn = M_arr[mask_BBN]
    dMdt = np.gradient(M_bbn, t_bbn)
    energy_rate = -dMdt * C_LIGHT**2
    energy_rate = np.maximum(energy_rate, 0.0)
    dt = np.gradient(t_bbn)
    cum_energy = np.cumsum(energy_rate * dt)
    total_energy = cum_energy[-1]
    if total_energy <= 0.0:
        return 1.0

    results = []

    mask_phot = t_bbn > t_SPLIT_S
    if np.any(mask_phot):
        E_phot = energy_rate[mask_phot] * dt[mask_phot]
        cum_E = np.cumsum(E_phot)
        idx = np.argmin(np.abs(cum_E - 0.5 * cum_E[-1]))
        t_med = t_bbn[mask_phot][idx]
        tau = t_med / 0.79
        limit = bbn_limit(tau)
        mY = frac * RHO_OVER_S_GEV
        f_max = limit / mY if mY > 0 else 1.0
        results.append(f_max)

    mask_had = t_bbn <= t_SPLIT_S
    if np.any(mask_had):
        E_had = energy_rate[mask_had] * dt[mask_had]
        cum_E = np.cumsum(E_had)
        idx = np.argmin(np.abs(cum_E - 0.5 * cum_E[-1]))
        t_med = t_bbn[mask_had][idx]
        tau = t_med / 0.71
        limit = bbn_limit(tau)
        mY = frac * RHO_OVER_S_GEV
        f_max = limit / mY if mY > 0 else 1.0
        results.append(f_max)

    if len(results) == 0:
        return 1.0
    return min(1.0, max(1e-30, float(np.min(results))))

print("Custom evolution and constraint functions defined.")


### Four Transitions Comparison

CMB+BBN combined constraints for all $4 \times 2 = 8$ combinations.


In [ ]:
# ==================== Four Transitions Comparison Plot ====================
M_vals = np.logspace(4, 14, 20)
q = 0.5
delta = 0.1
k = 2

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
fig.suptitle(
    f'Four Transition Functions: CMB+BBN Combined Constraints\n'
    f'(q={q}, delta={delta}, k={k})',
    fontsize=16, y=0.98)

# Default main-paper prescription (tanh + multiplicative)
f_default = []
for M in M_vals:
    fc = compute_f_PBH_limit(M, q, delta, k)[0]
    fb = compute_bbn_constraint(M, q, delta, k)
    f_default.append(min(fc, fb))
f_default = np.array(f_default)

for idx, (h_name, h_func) in enumerate(TRANSITION_FUNCTIONS.items()):
    ax = axes[idx // 2, idx % 2]

    colors = {'additive': 'blue', 'multiplicative': 'red'}
    linestyles = {'additive': '-', 'multiplicative': '--'}

    for dist in DISTRIBUTIONS:
        f_comb = []
        for M in M_vals:
            fc = compute_f_PBH_limit(M, q, delta, k)[0]
            fb = get_custom_bbn(M, q, delta, k, h_func, dist)
            f_comb.append(min(fc, fb))
        f_comb = np.array(f_comb)

        ax.loglog(M_vals, f_comb,
                  color=colors[dist], ls=linestyles[dist], lw=2.5,
                  label=f'{dist}')

    # Default reference curve
    ax.loglog(M_vals, f_default, 'k:', lw=2.0,
              label='Default (tanh+mult)')

    ax.set_title(f'{h_name}', fontsize=14)
    ax.set_xlabel(r'$M_i$ [g]', fontsize=12)
    ax.set_ylabel(r'$f_{\rm PBH,0}$', fontsize=12)
    ax.set_xlim(1e4, 1e14)
    ax.set_ylim(1e-20, 2.0)
    ax.legend(fontsize=10, loc='lower left')
    ax.grid(True, which='both', ls='--', alpha=0.4)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()


### Abundance Enhancement Ratio

$f_{\rm new} / f_{\rm default}$ for each prescription. Ratio $> 1$ means a looser constraint (more PBHs allowed).


In [ ]:
# ==================== Abundance Enhancement Ratio ====================
M_vals = np.logspace(4, 14, 20)
q = 0.5
delta = 0.1
k = 2

# Default
f_default = []
for M in M_vals:
    fc = compute_f_PBH_limit(M, q, delta, k)[0]
    fb = compute_bbn_constraint(M, q, delta, k)
    f_default.append(min(fc, fb))
f_default = np.array(f_default)

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
fig.suptitle(
    f'Abundance Enhancement Ratio  $f_{{prescription}} / f_{{default}}$\n'
    f'(q={q}, delta={delta}, k={k};  ratio>1 means looser constraint)',
    fontsize=16, y=0.98)

for idx, (h_name, h_func) in enumerate(TRANSITION_FUNCTIONS.items()):
    ax = axes[idx // 2, idx % 2]

    colors = {'additive': 'blue', 'multiplicative': 'red'}
    linestyles = {'additive': '-', 'multiplicative': '--'}

    for dist in DISTRIBUTIONS:
        f_comb = []
        for M in M_vals:
            fc = compute_f_PBH_limit(M, q, delta, k)[0]
            fb = get_custom_bbn(M, q, delta, k, h_func, dist)
            f_comb.append(min(fc, fb))
        f_comb = np.array(f_comb)

        ratio = f_comb / f_default
        ratio = np.clip(ratio, 1e-3, 1e3)

        ax.semilogx(M_vals, ratio,
                    color=colors[dist], ls=linestyles[dist], lw=2.5,
                    label=f'{dist}')

    ax.axhline(1.0, color='black', ls=':', lw=1.5, label='Default=1')
    ax.set_title(f'{h_name}', fontsize=14)
    ax.set_xlabel(r'$M_i$ [g]', fontsize=12)
    ax.set_ylabel(r'$f_{\rm new} / f_{\rm default}$', fontsize=12)
    ax.set_xlim(1e4, 1e14)
    ax.set_ylim(0.1, 10.0)
    ax.legend(fontsize=10, loc='upper left')
    ax.grid(True, which='both', ls='--', alpha=0.4)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

print("\nAll computations and plots completed!")
